# One-dimensional maps — exercise lab

This laboratory accompanies [the chapter](https://mvreeuwijk.github.io/chaosbook/python/disc1d.html). It runs
entirely in your browser (Python via Pyodide) — nothing to install, and any
changes you make are yours alone: re-opening the lab from the chapter resets it.

The full text of the exercises is in the
[chapter's exercise section](https://mvreeuwijk.github.io/chaosbook/python/disc1d.html#exercises); this notebook
gives you a running start on each one, and the exercise data sets are already
in its filesystem under `data/`.

In [ ]:
# Run this cell first: it installs the book's package into the
# in-browser Python (takes a few seconds, needs to run once per visit).
%pip install -q chaosbook ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider
import chaosbook as cb

## The logistic map at full parameter freedom

The chapter's interactive figures step through a precomputed grid of $r$
values. Here every slider move iterates the map afresh: series, cobweb and
return plot for any $r$ and any $x_0$.

In [ ]:
def logistic_explorer(r=3.5, x0=0.2, n=100):
    X = cb.orbit(cb.logistic, x0, n, r=r)
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 3.2))
    ax1.plot(X, "k.-", linewidth=0.5, markersize=3)
    ax1.set_xlabel("$n$")
    ax1.set_ylabel("$x_n$")
    cb.cobweb(cb.logistic, x0, min(n, 50), ax=ax2, r=r)
    ax3.plot(X[:-1], X[1:], "k.", markersize=3)
    ax3.set_xlabel("$x_n$")
    ax3.set_ylabel("$x_{n+1}$")
    fig.tight_layout()
    plt.show()

interact(logistic_explorer,
         r=FloatSlider(3.5, min=0.0, max=4.0, step=0.01),
         x0=FloatSlider(0.2, min=0.0, max=1.0, step=0.01),
         n=(10, 500, 10));

In [ ]:
# Zoom anywhere in the bifurcation diagram and the Lyapunov exponent
# Lambda(r) below it - the chapter's figure had a fixed grid; here you
# choose the window. Try the period-3 window rmin, rmax = 3.82, 3.87.
def dlogistic(x, r):
    return r * (1 - 2 * x)

rmin, rmax = 2.9, 4.0
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
cb.bifurcation_diagram(cb.logistic, rmin, rmax, nr=400, n=600, ax=ax1)
rs, lams = cb.lyapunov_sweep(cb.logistic, dlogistic, rmin, rmax, nr=400)
ax2.plot(rs, lams, "k", linewidth=0.7)
ax2.axhline(0, color="gray", linewidth=0.7)
ax2.set_xlabel("$r$")
ax2.set_ylabel("$\\Lambda(r)$")
fig.tight_layout()
plt.show()

## Exercise: classification of timeseries

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/python/disc1d.html#exercises). The three mystery
series are already here — no download needed.

In [ ]:
set1 = np.loadtxt("data/set1.txt")
set2 = np.loadtxt("data/set2.txt")
set3 = np.loadtxt("data/set3.txt")
plt.figure(figsize=(8, 3))
plt.plot(set1, "k.-", linewidth=0.5, markersize=2)
plt.xlabel("$n$")
plt.ylabel("$x_n$")
plt.show()

In [ ]:
# A return plot reveals more than the series: which set is pure noise,
# which a one-dimensional mapping, which a higher-order mapping?
X = set1
plt.plot(X[:-1], X[1:], "k.", markersize=3)
plt.xlabel("$x_n$")
plt.ylabel("$x_{n+1}$")
plt.show()

## Exercise: Newton-Raphson method

Full text: [parts a)–f) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/disc1d.html#exercises).
The method is itself a one-dimensional map, so `cb.orbit` iterates it.

In [ ]:
# a) g(x) = 3 tanh(x) - x^3: iterate f(x) = x - g(x)/g'(x) from x0 = 1.0
def g(x):
    return 3 * np.tanh(x) - x**3

def dgdx(x):
    return 3 / np.cosh(x)**2 - 3 * x**2

def f(x):
    return x - g(x) / dgdx(x)

X = cb.orbit(f, 1.0, 20)
plt.plot(X, "k.-")
plt.xlabel("$n$")
plt.ylabel("$x_n$")
plt.show()
print("x* =", X[-1], "   g(x*) =", g(X[-1]))

In [ ]:
# d)-f): the function g(x) = [cosh(4 - 3/x) - 1]^(1/a).
# Your turn: write down g'(x), build the Newton-Raphson map f2 as above,
# iterate from x0 = 0.4 until N = 100, and show the series and return plot.
# For f) sweep a in [2, 6] and collect the late iterates into a
# bifurcation diagram.
a = 6

def g2(x):
    return (np.cosh(4 - 3 / x) - 1) ** (1 / a)

## Exercise: cubic map

Full text: [parts a)–g) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/disc1d.html#exercises).
`cb.orbit` and `cb.cobweb` take *any* map $f(x, \ldots)$ — the package never
assumes the built-in ones.

In [ ]:
# The cubic map x_{n+1} = lam * x * (1 - x^2/lam), 0 < lam < 3.
def cubic(x, lam=1.5):
    return lam * x * (1 - x**2 / lam)

# b)-d): the cobweb shows which initial conditions stay finite and which
# fixed point attracts them - vary x0 and lam.
cb.cobweb(cubic, 0.3, 40, xmin=-2.0, xmax=2.0, lam=1.5)
plt.show()

In [ ]:
# e)-g): hunt for period-2 (and period-3) solutions - a series plot makes
# a periodic trajectory obvious.
X = cb.orbit(cubic, 0.3, 60, lam=2.2)
plt.plot(X, "k.-", linewidth=0.5)
plt.xlabel("$n$")
plt.ylabel("$x_n$")
plt.show()

## Exercise: universality revisited

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/python/disc1d.html#exercises).

In [ ]:
# a) the family x_{n+1} = r (1 - |2x - 1|^eta) on [0, 1]:
# for which eta does the period-doubling route stay universal?
def hump(x, r, eta=2.0):
    return r * (1 - np.abs(2 * x - 1) ** eta)

for eta in (2.0, 0.75):
    plt.figure(figsize=(6, 4))
    cb.bifurcation_diagram(lambda x, r: hump(x, r, eta=eta), 0.0, 1.0,
                           nr=400, n=600)
    plt.title(f"$\\eta = {eta}$")
    plt.show()
# Your turn: other values of eta - what about eta = 10?

In [ ]:
# b) x_{n+1} = 3.5 r (1 - r) sin(pi x): note that here the *parameter*
# enters through 3.5 r (1 - r) - discuss what you see.
def sinfam(x, r):
    return 3.5 * r * (1 - r) * np.sin(np.pi * x)

cb.bifurcation_diagram(sinfam, 0.0, 1.0, nr=400, n=600)
plt.show()

## Exercise: discrete cubic map

Full text: [parts a)–g) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/disc1d.html#exercises).

In [ ]:
# a) x_{n+1} = r x (3 - 4 x^2) at r = 0.97: series and return plot
# from x0 = 0.2, N = 1000.
def dcubic(x, r=0.97):
    return r * x * (3 - 4 * x**2)

X = cb.orbit(dcubic, 0.2, 1000, r=0.97)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
ax1.plot(X, "k.", markersize=2)
ax1.set_xlabel("$n$")
ax1.set_ylabel("$x_n$")
ax2.plot(X[:-1], X[1:], "k.", markersize=2)
ax2.set_xlabel("$x_n$")
ax2.set_ylabel("$x_{n+1}$")
fig.tight_layout()
plt.show()

In [ ]:
# e) the bifurcation diagram over r in [0, 1] (x0 in [-1, 1]).
cb.bifurcation_diagram(dcubic, 0.0, 1.0, nr=400, n=600, x0=0.2)
plt.show()

In [ ]:
# f)-g): two nearby series at r = 1, and the Lyapunov exponent -
# compare cb.lyapunov's estimate with the closed form's ln 3.
def ddcubic(x, r=1.0):
    return r * (3 - 12 * x**2)

X1 = cb.orbit(dcubic, 0.2, 50, r=1.0)
X2 = cb.orbit(dcubic, 0.2 + 1e-6, 50, r=1.0)
plt.plot(X1, "b.-", linewidth=0.5)
plt.plot(X2, "r.-", linewidth=0.5)
plt.xlabel("$n$")
plt.ylabel("$x_n$")
plt.show()
print("Lambda =", cb.lyapunov(dcubic, ddcubic, 0.2, r=1.0),
      "   ln 3 =", np.log(3))

---
*Back to [the chapter](https://mvreeuwijk.github.io/chaosbook/python/disc1d.html) — or open the
[cookbook](cookbook.ipynb) in this same lab environment.*